In [ ]:
!pip install onnxruntime
!pip install ultralytics
!pip install tensorrt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Importando as bibliotecas
import os
import numpy as np
import psutil
import pynvml
import time
import tensorflow as tf
import pandas as pd
import tensorrt as trt
import cv2
from threading import Thread, Event
from queue import Queue

ModuleNotFoundError: No module named 'tensorrt'

In [ ]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.4/97.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.6/105.6 kB 12.1 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2025.1.1-cp311-cp311-linux_x86_64.whl size=660712 sha256=0807fb666b3390952566b5ccea9ff29bf515e12cad0c5eea8916a575b750e8a1
  Stored in directory: /root/.cache/pip/wheels/49/0a/64/6530a5fde64f984ebb4992e38744fdfd2a61f510377b3a24d9
Successfully built pycuda


In [ ]:
## Instalando as bibliotecas do roboflow
## importing required libraries
import os
import shutil
import random
!pip install tqdm --upgrade
from tqdm.notebook import tqdm
from IPython import display
display.clear_output()
from ultralytics import YOLO
from IPython.display import display, Image
import ultralytics
ultralytics.checks()
%cd /content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas
!ls
!nvidia-smi
import os
HOME = os.getcwd()
print(HOME)
!mkdir {HOME}/datasets2
%cd {HOME}/datasets2

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="aJO0DguOcxOKKsVf2mq5")
project = rf.workspace("alvaro-z5qmu").project("floresta-wgiln")
version = project.version(3)
dataset = version.download("yolov8")






In [ ]:
#medindo so inferencia
import os
import time
import numpy as np
import psutil
import pynvml
import pandas as pd
import torch
from ultralytics import YOLO
from PIL import Image
import gc

# ==== CONFIGURAÇÕES ====
CONFIG = {
    "conf_threshold": 0.3,
    "iou_threshold": 0.5,
    "max_images": 16,
    "image_folder": "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/datasets2/Floresta-3/valid/images",
    "model_paths": [
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov8/yolov8s/best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov8/yolov8n/best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov8/yolov8m/best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov8/yolov8L/best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov8/yolov8x/best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov11/yolov11s/Copy of best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov11/Yolov11n/Copy of best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov11/yolov11m/Copy of best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov11/Yolov11L/best.pt",
        "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/ModelosOriginais/Yolov11/yolov11x/Copy of best.pt",
    ],
    "salvar_saida_txt": True,
    "saida_txt_path": "saida_metricas2.txt",
    "saida_csv_path": "saida_metricas2.csv"
}

# ==== FUNÇÕES UTILITÁRIAS ====
def get_ram(proc):
    return proc.memory_info().rss / (1024 ** 2)  # MB

# Inicializa medição de GPU
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def format_saida(index, model_name, model_size, num_images, total_time, fps, ram_cpu, ram_gpu, cpu, energy, avg_power, duration):
    media_por_img = total_time / num_images
    return f"""
======== 📊 MÉTRICAS DO MODELO {index}: {model_name} ========

📦 Tamanho do modelo           : {model_size:.2f} MB
🖼️  Número de imagens          : {num_images}

⏱️  Tempo total de inferência  : {total_time:.2f} segundos
⏱️  Tempo médio por imagem     : {media_por_img:.4f} segundos
⚡ FPS (frames por segundo)    : {fps:.2f}

🧠 RAM CPU média usada         : {ram_cpu:.2f} MB
🧠 RAM GPU pico usado          : {ram_gpu:.2f} MB
⚙️  CPU média utilizada        : {cpu:.2f} %

⚡ Energia total da GPU        : {energy:.2f} Joules
⚡ Potência média da GPU       : {avg_power:.2f} Watts
🕒 Duração da medição (GPU)    : {duration:.2f} segundos

============================================================
"""

def test_yolo_metrics(model_path, image_paths, index, conf=0.3, iou=0.5):
    print(f"\n🔍 Avaliando modelo {index}/{len(CONFIG['model_paths'])}: {os.path.basename(model_path)}")
    proc = psutil.Process(os.getpid())
    model = YOLO(model_path)
    model.fuse()
    model_size = os.path.getsize(model_path) / (1024 * 1024)

    ram_usos, cpu_usos, durations, energies, powers = [], [], [], [], []
    total_inf_time = 0

    torch.cuda.reset_peak_memory_stats()

    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")

        psutil.cpu_percent(interval=None)
        ram = get_ram(proc)
        start_power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
        t0 = time.time()

        with torch.inference_mode():
            _ = model(img, verbose=False)

        t1 = time.time()
        end_power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
        cpu = psutil.cpu_percent(interval=None)

        duration = t1 - t0
        avg_power = (start_power + end_power) / 2
        energy = avg_power * duration

        ram_usos.append(ram)
        cpu_usos.append(cpu)
        durations.append(duration)
        energies.append(energy)
        powers.append(avg_power)
        total_inf_time += duration

    fps = len(image_paths) / total_inf_time
    ram_gpu_peak = torch.cuda.max_memory_allocated() / (1024 ** 2)

    saida_texto = format_saida(
        index, os.path.basename(model_path), model_size, len(image_paths),
        total_inf_time, fps,
        np.mean(ram_usos), ram_gpu_peak, np.mean(cpu_usos),
        sum(energies), np.mean(powers), sum(durations)
    )

    saida_csv = {
        "Modelo": os.path.basename(model_path),
        "Tamanho_MB": round(model_size, 2),
        "Num_Imagens": len(image_paths),
        "Tempo_Segundos": round(total_inf_time, 2),
        "Tempo_Medio_Imagem": round(total_inf_time / len(image_paths), 4),
        "FPS": round(fps, 2),
        "RAM_CPU_MB": round(np.mean(ram_usos), 2),
        "RAM_GPU_MB": round(ram_gpu_peak, 2),
        "CPU_%": round(np.mean(cpu_usos), 2),
        "Energia_J": round(sum(energies), 2),
        "Potencia_W": round(np.mean(powers), 2),
        "Duracao_GPU_s": round(sum(durations), 2)
    }

    return saida_texto, saida_csv, model

# ==== EXECUÇÃO PRINCIPAL ====
if __name__ == "__main__":
    try:
        image_paths = sorted([
            os.path.join(CONFIG["image_folder"], f)
            for f in os.listdir(CONFIG["image_folder"])
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])[:CONFIG["max_images"]]

        saidas_txt = []
        saidas_csv = []

        for idx, model_path in enumerate(CONFIG["model_paths"], start=1):
            output_txt, output_csv, model = test_yolo_metrics(
                model_path,
                image_paths,
                index=idx,
                conf=CONFIG["conf_threshold"],
                iou=CONFIG["iou_threshold"]
            )
            print(output_txt)
            saidas_txt.append(output_txt)
            saidas_csv.append(output_csv)

            # 🔁 Limpeza após cada modelo
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            del model
            gc.collect()

        if CONFIG["salvar_saida_txt"]:
            with open(CONFIG["saida_txt_path"], "w") as f:
                f.write("\n\n".join(saidas_txt))
            print(f"\n📁 Resultados salvos em: {CONFIG['saida_txt_path']}")

            df = pd.DataFrame(saidas_csv)
            df.to_csv(CONFIG["saida_csv_path"], index=False)
            print(f"📁 Resultados salvos em CSV: {CONFIG['saida_csv_path']}")

    finally:
        pynvml.nvmlShutdown()



🔍 Avaliando modelo 1/10: best.pt
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 42.4 GFLOPs

======== 📊 MÉTRICAS DO MODELO 1: best.pt ========

📦 Tamanho do modelo           : 22.78 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 0.49 segundos
⏱️  Tempo médio por imagem     : 0.0308 segundos
⚡ FPS (frames por segundo)    : 32.44

🧠 RAM CPU média usada         : 2740.57 MB
🧠 RAM GPU pico usado          : 590.73 MB
⚙️  CPU média utilizada        : 25.25 %

⚡ Energia total da GPU        : 22.59 Joules
⚡ Potência média da GPU       : 46.14 Watts
🕒 Duração da medição (GPU)    : 0.49 segundos



🔍 Avaliando modelo 2/10: best.pt
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 12.0 GFLOPs

======== 📊 MÉTRICAS DO MODELO 2: best.pt ========

📦 Tamanho do modelo           : 12.81 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 0.26 segundos
⏱️  Tempo médio por imagem     : 0.0163 segundos
⚡ 

## Detectron

In [ ]:
!python -m pip install roboflow
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-n1f540fj
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-n1f540fj
  Resolved https://github.com/facebookresearch/detectron2.git to commit 18f69583391e5040043ca4f4bebd2c60f0ebfde0
  Preparing metadata (setup.py) ... done


In [ ]:
import os
import time
import numpy as np
import psutil
import pynvml
import cv2
import torch
import gc
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.model_zoo import model_zoo
from detectron2.modeling import build_model

CONFIG = {
    "conf_threshold": 0.3,
    "max_images": 16,
    "image_folder": "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/datasets2/Floresta-3/valid/images",
    "models": [
        {
            "name": "COCO-InstanceSegmentation/mask_rcnn_R_101_C4_3x.yaml",
            "path": "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_101_C4_3x/2025-02-18-16-56-18/model_final.pth"
        },
        {
            "name": "COCO-InstanceSegmentation/mask_rcnn_R_101_DC5_3x.yaml",
            "path": "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_101_DC5_3x/2025-02-18-17-09-55/model_final.pth"
        },
        {
            "name": "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml",
            "path": "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_101_FPN_3x/2025-02-18-17-14-52/model_final.pth"
        },
        {
            "name": "COCO-InstanceSegmentation/mask_rcnn_R_50_C4_1x.yaml",
            "path": "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_50_C4_1x/2025-02-18-17-18-03/model_final.pth"
        },
        {
            "name": "COCO-InstanceSegmentation/mask_rcnn_R_50_C4_3x.yaml",
            "path": "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_50_C4_3x/2025-02-18-17-28-59/model_final.pth"
        },
        {
            "name": "COCO-InstanceSegmentation/mask_rcnn_R_50_DC5_1x.yaml",
            "path": "/content/drive/MyDrive/AlvaroSampaio/DetectronTeste2/Floresta/mask_rcnn_R_50_DC5_1x/2025-02-19-14-08-33/model_final.pth"
        }
    ],
    "salvar_saida_txt": True,
    "saida_txt_path": "saida_metricas_detectron.txt"
}
def get_ram(proc):
    return proc.memory_info().rss / (1024 ** 2)  # MB

def format_saida(index, model_name, model_size, num_images, total_time, fps, ram, cpu, energy, avg_power, duration, ram_gpu_peak):
    media_por_img = total_time / num_images
    return f"""
======== 📊 MÉTRICAS DO MODELO {index}: {model_name} ========

📦 Tamanho do modelo           : {model_size:.2f} MB
🖼️  Número de imagens          : {num_images}

⏱️  Tempo total de inferência  : {total_time:.2f} segundos
⏱️  Tempo médio por imagem     : {media_por_img:.4f} segundos
⚡ FPS (frames por segundo)    : {fps:.2f}

🧠 RAM média usada (CPU)       : {ram:.2f} MB
⚙️  CPU média utilizada        : {cpu:.2f} %
🧠 Pico de RAM da GPU          : {ram_gpu_peak:.2f} MB

⚡ Energia total da GPU        : {energy:.2f} Joules
⚡ Potência média da GPU       : {avg_power:.2f} Watts
🕒 Duração da medição (GPU)    : {duration:.2f} segundos

============================================================
"""

def load_model(cfg_file, weights_path, conf_threshold):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(cfg_file))
    cfg.MODEL.WEIGHTS = weights_path
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = conf_threshold
    cfg.MODEL.DEVICE = "cuda"
    model = build_model(cfg)
    DetectionCheckpointer(model).load(weights_path)
    model.eval()
    return model, cfg

def test_detectron_model(cfg_file, weights_path, image_paths, index):
    print(f"🔍 Avaliando modelo {index}: {cfg_file}")
    proc = psutil.Process(os.getpid())
    model, cfg = load_model(cfg_file, weights_path, CONFIG["conf_threshold"])
    model_size = os.path.getsize(weights_path) / (1024 * 1024)
    torch.cuda.reset_peak_memory_stats()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    ram_usos, cpu_usos, durations, energies, powers = [], [], [], [], []
    total_inf_time = 0

    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            continue
        batched_input = [{"image": torch.from_numpy(img).permute(2, 0, 1).float().cuda()}]

        psutil.cpu_percent(interval=None)
        ram = get_ram(proc)
        start_power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
        t0 = time.time()

        with torch.inference_mode():
            _ = model(batched_input)

        t1 = time.time()
        end_power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
        cpu = psutil.cpu_percent(interval=None)

        duration = t1 - t0
        avg_power = (start_power + end_power) / 2
        energy = avg_power * duration

        ram_usos.append(ram)
        cpu_usos.append(cpu)
        durations.append(duration)
        energies.append(energy)
        powers.append(avg_power)
        total_inf_time += duration

    ram_gpu_peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    fps = len(image_paths) / total_inf_time if total_inf_time > 0 else 0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    del model
    gc.collect()

    return format_saida(index, cfg_file, model_size, len(image_paths), total_inf_time,
                        fps, np.mean(ram_usos), np.mean(cpu_usos), sum(energies),
                        np.mean(powers), sum(durations), ram_gpu_peak)

# ==== EXECUÇÃO ====
if __name__ == "__main__":
    pynvml.nvmlInit()
    try:
        image_paths = sorted([
            os.path.join(CONFIG["image_folder"], f)
            for f in os.listdir(CONFIG["image_folder"])
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])[:CONFIG["max_images"]]

        saidas = []
        for idx, model in enumerate(CONFIG["models"], 1):
            saida = test_detectron_model(model["name"], model["path"], image_paths, idx)
            print(saida)
            saidas.append(saida)

        if CONFIG["salvar_saida_txt"]:
            with open(CONFIG["saida_txt_path"], "w") as f:
                f.write("\n\n".join(saidas))
            print(f"📁 Resultados salvos em: {CONFIG['saida_txt_path']}")
    finally:
        pynvml.nvmlShutdown()


🔍 Avaliando modelo 1: COCO-InstanceSegmentation/mask_rcnn_R_101_C4_3x.yaml


roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}
/usr/local/lib/python3.11/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



======== 📊 MÉTRICAS DO MODELO 1: COCO-InstanceSegmentation/mask_rcnn_R_101_C4_3x.yaml ========

📦 Tamanho do modelo           : 412.46 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 6.82 segundos
⏱️  Tempo médio por imagem     : 0.4260 segundos
⚡ FPS (frames por segundo)    : 2.35

🧠 RAM média usada (CPU)       : 2069.38 MB
⚙️  CPU média utilizada        : 19.44 %
🧠 Pico de RAM da GPU          : 2244.71 MB

⚡ Energia total da GPU        : 435.25 Joules
⚡ Potência média da GPU       : 64.78 Watts
🕒 Duração da medição (GPU)    : 6.82 segundos


🔍 Avaliando modelo 2: COCO-InstanceSegmentation/mask_rcnn_R_101_DC5_3x.yaml


roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}



======== 📊 MÉTRICAS DO MODELO 2: COCO-InstanceSegmentation/mask_rcnn_R_101_DC5_3x.yaml ========

📦 Tamanho do modelo           : 1456.46 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 2.46 segundos
⏱️  Tempo médio por imagem     : 0.1538 segundos
⚡ FPS (frames por segundo)    : 6.50

🧠 RAM média usada (CPU)       : 2311.10 MB
⚙️  CPU média utilizada        : 18.48 %
🧠 Pico de RAM da GPU          : 1177.54 MB

⚡ Energia total da GPU        : 172.08 Joules
⚡ Potência média da GPU       : 70.52 Watts
🕒 Duração da medição (GPU)    : 2.46 segundos


🔍 Avaliando modelo 3: COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml


...


======== 📊 MÉTRICAS DO MODELO 4: COCO-InstanceSegmentation/mask_rcnn_R_50_C4_1x.yaml ========

📦 Tamanho do modelo           : 267.47 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 4.83 segundos
⏱️  Tempo médio por imagem     : 0.3020 segundos
⚡ FPS (frames por segundo)    : 3.31

🧠 RAM média usada (CPU)       : 2256.14 MB
⚙️  CPU média utilizada        : 16.92 %
🧠 Pico de RAM da GPU          : 2172.18 MB

⚡ Energia total da GPU        : 322.07 Joules
⚡ Potência média da GPU       : 66.63 Watts
🕒 Duração da medição (GPU)    : 4.83 segundos


🔍 Avaliando modelo 5: COCO-InstanceSegmentation/mask_rcnn_R_50_C4_3x.yaml


roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}



======== 📊 MÉTRICAS DO MODELO 5: COCO-InstanceSegmentation/mask_rcnn_R_50_C4_3x.yaml ========

📦 Tamanho do modelo           : 267.47 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 5.10 segundos
⏱️  Tempo médio por imagem     : 0.3190 segundos
⚡ FPS (frames por segundo)    : 3.14

🧠 RAM média usada (CPU)       : 2256.14 MB
⚙️  CPU média utilizada        : 19.85 %
🧠 Pico de RAM da GPU          : 2267.89 MB

⚡ Energia total da GPU        : 338.61 Joules
⚡ Potência média da GPU       : 66.44 Watts
🕒 Duração da medição (GPU)    : 5.10 segundos


🔍 Avaliando modelo 6: COCO-InstanceSegmentation/mask_rcnn_R_50_DC5_1x.yaml


roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}



======== 📊 MÉTRICAS DO MODELO 6: COCO-InstanceSegmentation/mask_rcnn_R_50_DC5_1x.yaml ========

📦 Tamanho do modelo           : 1311.47 MB
🖼️  Número de imagens          : 16

⏱️  Tempo total de inferência  : 2.09 segundos
⏱️  Tempo médio por imagem     : 0.1303 segundos
⚡ FPS (frames por segundo)    : 7.67

🧠 RAM média usada (CPU)       : 2256.14 MB
⚙️  CPU média utilizada        : 19.04 %
🧠 Pico de RAM da GPU          : 1104.02 MB

⚡ Energia total da GPU        : 146.41 Joules
⚡ Potência média da GPU       : 70.71 Watts
🕒 Duração da medição (GPU)    : 2.09 segundos


📁 Resultados salvos em: saida_metricas_detectron.txt


## U-net

In [ ]:
# Corrige o erro de inicialização do cuDNN
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


In [ ]:
import os
import time
import numpy as np
import psutil
import pynvml
from PIL import Image
import tensorflow as tf
import pandas as pd

# ==== CONFIGURAÇÕES ====
CONFIG = {
    "max_images": 16,
    "image_folder": "/content/drive/MyDrive/AlvaroSampaio/ExtracaoMetricas/datasets2/Floresta-3/valid/images",
    "model_paths": [
        "/content/drive/MyDrive/AlvaroSampaio/u-net/unet_instance_segmentation_best.keras"
    ],
    "image_size": (256, 256),
    "salvar_saida_txt": True,
    "saida_txt_path": "saida_metricas_unet.txt",
    "saida_csv_path": "saida_metricas_unet.csv"
}

# ==== FUNÇÕES ====
def get_ram(proc):
    return proc.memory_info().rss / (1024 ** 2)  # MB

def get_gpu_ram_peak(handle):
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return mem_info.used / (1024 ** 2)  # MB

def measure_gpu_energy(handle):
    power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
    timestamp = time.time()
    return power, timestamp

def finalize_gpu_energy(handle, start_power, start_time):
    end_power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
    end_time = time.time()
    duration = end_time - start_time
    avg_power = (start_power + end_power) / 2
    energy = avg_power * duration
    return energy, avg_power, duration

def preprocess_image(img_path, target_size):
    img = Image.open(img_path).convert("RGB").resize(target_size)
    img_array = np.array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

def test_unet_metrics(model_path, image_paths, index, image_size, handle):
    print(f"\n🔍 Avaliando modelo {index}: {os.path.basename(model_path)}")
    proc = psutil.Process(os.getpid())
    model = tf.keras.models.load_model(model_path)
    model_size = os.path.getsize(model_path) / (1024 * 1024)

    ram_usos, cpu_usos, durations, powers, energies = [], [], [], [], []
    ram_gpu_peak = 0
    total_inf_time = 0

    for i, img_path in enumerate(image_paths):
        input_img = preprocess_image(img_path, image_size)

        psutil.cpu_percent(interval=None)
        ram = get_ram(proc)
        start_power, t0 = measure_gpu_energy(handle)

        _ = model.predict(input_img, verbose=0)

        t1 = time.time()
        end_power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0
        cpu = psutil.cpu_percent(interval=None)
        ram_gpu_now = get_gpu_ram_peak(handle)

        duration = t1 - t0
        avg_power = (start_power + end_power) / 2
        energy = avg_power * duration

        ram_usos.append(ram)
        cpu_usos.append(cpu)
        durations.append(duration)
        powers.append(avg_power)
        energies.append(energy)
        ram_gpu_peak = max(ram_gpu_peak, ram_gpu_now)
        total_inf_time += duration

        print(f"✅ Imagem {i+1}/{len(image_paths)} - Tempo: {duration:.4f}s")

    fps = len(image_paths) / total_inf_time if total_inf_time > 0 else 0

    saida_csv = {
        "Modelo": os.path.basename(model_path),
        "Tamanho_MB": round(model_size, 2),
        "Num_Imagens": len(image_paths),
        "Tempo_Segundos": round(total_inf_time, 2),
        "Tempo_Medio_Imagem": round(total_inf_time / len(image_paths), 4),
        "FPS": round(fps, 2),
        "RAM_CPU_MB": round(np.mean(ram_usos), 2),
        "RAM_GPU_MB": round(ram_gpu_peak, 2),
        "CPU_%": round(np.mean(cpu_usos), 2),
        "Energia_J": round(sum(energies), 2),
        "Potencia_W": round(np.mean(powers), 2),
        "Duracao_GPU_s": round(sum(durations), 2)
    }

    return saida_csv

# ==== EXECUÇÃO ====
if __name__ == "__main__":
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    try:
        image_paths = sorted([
            os.path.join(CONFIG["image_folder"], f)
            for f in os.listdir(CONFIG["image_folder"])
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])[:CONFIG["max_images"]]

        resultados = []
        for idx, model_path in enumerate(CONFIG["model_paths"], 1):
            resultado = test_unet_metrics(model_path, image_paths, idx, CONFIG["image_size"], handle)
            resultados.append(resultado)

        df = pd.DataFrame(resultados)
        df.to_csv(CONFIG["saida_csv_path"], index=False)
        print(f"\n📁 CSV salvo em: {CONFIG['saida_csv_path']}")

    finally:
        pynvml.nvmlShutdown()



🔍 Avaliando modelo 1: unet_instance_segmentation_best.keras
✅ Imagem 1/16 - Tempo: 4.2565s
✅ Imagem 2/16 - Tempo: 0.0720s
✅ Imagem 3/16 - Tempo: 0.0796s
✅ Imagem 4/16 - Tempo: 0.0804s
✅ Imagem 5/16 - Tempo: 0.0811s
✅ Imagem 6/16 - Tempo: 0.0771s
✅ Imagem 7/16 - Tempo: 0.0809s
✅ Imagem 8/16 - Tempo: 0.0790s
✅ Imagem 9/16 - Tempo: 0.1030s
✅ Imagem 10/16 - Tempo: 0.0870s
✅ Imagem 11/16 - Tempo: 0.0793s
✅ Imagem 12/16 - Tempo: 0.0815s
✅ Imagem 13/16 - Tempo: 0.0796s
✅ Imagem 14/16 - Tempo: 0.0888s
✅ Imagem 15/16 - Tempo: 0.0857s
✅ Imagem 16/16 - Tempo: 0.0816s

📁 CSV salvo em: saida_metricas_unet.csv
